# 2. Extract MetacellFormer cell embeddings

Run from the GitHub repository root with the **MetacellFormer-env** kernel. Download the MetacellFormer-evaluation dataset and MetacellFormer model. Set `METACELLFORMER_EVALUATION` and `METACELLFORMER_MODEL` to their directories, or edit the defaults below.

The released DC input has 569 cells and a fixed train/test roster. This notebook takes the first eight cells per class and split in roster order (64 cells total) and extracts native embeddings. The archived DC input is described as TPM by its source.

In [ ]:
from pathlib import Path
import json
import os
import sys
import subprocess
import anndata as ad
import numpy as np
from benchmarks.shared._common import roster, embeddings

repo = Path.cwd().resolve()
assert (repo / "benchmarks/embeddings/extract_embeddings.py").is_file(), "Start Jupyter from the repository root"
data_root = Path(os.environ.get("METACELLFORMER_EVALUATION", "../MetacellFormer-evaluation")).expanduser().resolve()
model_dir = Path(os.environ.get("METACELLFORMER_MODEL", "../MetacellFormer-model")).expanduser().resolve()
source = data_root / "inputs/cell_annotation/DC.h5ad"
source_roster = data_root / "contracts/cell_type_annotation/DC_roster.json"
assert source.is_file() and source_roster.is_file(), "Download the evaluation dataset"
assert (model_dir / "model.safetensors").is_file(), "Download the MetacellFormer model"
work = repo / "example_outputs/DC"
work.mkdir(parents=True, exist_ok=True)
config = work / "runtime.json"
config.write_text(json.dumps({"MetacellFormer": {"checkpoint": str(model_dir)}}, indent=2) + "\n")
device = "cpu"  # Change to cuda:0 if available.
full = roster(source_roster)
frame = full.groupby(["split", "label"], sort=False).head(8).reset_index(drop=True)
assert len(frame) == 64 and frame.groupby(["split", "label"]).size().eq(8).all()
roster_file = work / "example_roster.json"
roster_file.write_text(json.dumps({"obs_name": frame.cell_id.tolist(),
                                   "cell_type": frame.label.tolist(),
                                   "official_split": frame.split.tolist()}, indent=2) + "\n")
print("Using", len(frame), "fixed DC roster cells (8 per class and split)")

In [ ]:
backed = ad.read_h5ad(source, backed="r")
try:
    for split, name in [("0", "train"), ("2", "test")]:
        ids = frame.loc[frame.split.eq(split), "cell_id"].astype(str).to_numpy()
        positions = backed.obs_names.get_indexer(ids)
        assert len(ids) and (positions >= 0).all()
        part = backed[positions].to_memory()
        assert np.array_equal(part.obs_names.astype(str), ids)
        part.write_h5ad(work / f"{name}.h5ad")
finally:
    backed.file.close()

In [ ]:
for name in ("train", "test"):
    subprocess.run([sys.executable, "-m", "benchmarks.embeddings.extract_embeddings",
                    "--input", str(work / f"{name}.h5ad"),
                    "--output", str(work / f"{name}.npz"),
                    "--config", str(config), "--model", "MetacellFormer",
                    "--task", "cell", "--device", device, "--batch-size", "4"],
                   cwd=repo, check=True)
for split, name in [("0", "train"), ("2", "test")]:
    ids = frame.loc[frame.split.eq(split), "cell_id"].astype(str).to_numpy()
    values, returned, valid = embeddings(work / f"{name}.npz", ids)
    assert values.shape[0] == len(ids) and valid.all()
    print(name, values.shape, "valid", int(valid.sum()))